# Data Processing

This notebook converts raw simulations from `raw_data/` into stable plotting inputs in `processed_data/`. It is organized by the paper figures: Figure 3 for NK ruggedness inference, Figure 4 for empirical landscapes, and Figure 5 for strategy selection.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np

from slide.data_generation import RAW_FILENAMES, nk_grid_pairs
from slide.data_processing import (
    empirical_fourier_spectra,
    empirical_metric_comparison,
    fourier_analysis_summary,
    heterogeneity_data,
    load_landscapes,
    nk_de_summary,
    nk_metric_comparison_from_accuracy,
    load_raw,
    optimal_de_strategies,
    process_mutation_accuracy,
    process_popsize_accuracy,
    process_ruggedness_accuracy,
    save_processed,
    smooth_rugged_example,
    strategy_prediction_summary,
    subsampling_accuracy,
)
from slide.utils import get_raw_data_dir, get_processed_data_dir

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")


## NK Ruggedness Accuracy

Load NK decay and strategy sweeps for Figure 3A-B/E and the Figure 5A lookup-table analysis.

In [ ]:
strategy_grid = load_raw(RAW_FILENAMES["nk_strategy_grid"])
decay_grid = load_raw(RAW_FILENAMES["nk_decay_grid"])
nk_pairs = np.array(nk_grid_pairs())

ruggedness_accuracy = process_ruggedness_accuracy(decay_grid, nk_pairs)
save_processed(ruggedness_accuracy, "ruggedness_accuracy.pkl")


## Popsize And Mutation-Rate Accuracy

Process the robustness sweeps behind Figure 3C-D: population-size sensitivity and mutation-rate sensitivity of fitted ruggedness.

In [ ]:
popsize_data = load_raw(RAW_FILENAMES["nk_popsize_accuracy"])
mutation_data = load_raw(RAW_FILENAMES["nk_mutation_accuracy"])

save_processed(process_popsize_accuracy(popsize_data), "popsize_accuracy.pkl")
save_processed(process_mutation_accuracy(mutation_data), "mut_accuracy.pkl")


## Empirical Landscapes And Metrics

Load empirical GB1, TrpB, TEV, and ParD3 landscapes and compute full-landscape comparison metrics for Figure 4A/G.

In [ ]:
landscapes = load_landscapes()
empirical_decay_uniform = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_uniform"])
    for name in ("GB1", "TrpB", "TEV", "ParD3")
}
empirical_decay_all = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_all"])
    for name in ("GB1", "TrpB", "TEV", "ParD3")
}

save_processed(empirical_metric_comparison(landscapes, empirical_decay_all), "empirical_ruggedness_metric_comparison.pkl")
save_processed(empirical_fourier_spectra(landscapes), "fourier_spectra_empirical.pkl")


## Heterogeneity And Subsampling

Summarize local-vs-global empirical ruggedness and starting-point subsampling analyses for Figure 4B-F.

In [ ]:
nk_heterogeneity = load_raw(RAW_FILENAMES["nk_heterogeneity"])
save_processed(heterogeneity_data(nk_heterogeneity, empirical_decay_all), "heterogeneity_data.pkl")
save_processed(heterogeneity_data(nk_heterogeneity, empirical_decay_all, method="IK"), "heterogeneity_data_IK.pkl")

save_processed(subsampling_accuracy(empirical_decay_all), "trajectory_subsampling.pkl")
save_processed(subsampling_accuracy(empirical_decay_all, method="IK"), "trajectory_subsampling_IK.pkl")


## Optimal Strategies

Connect fitted decay rates to optimal directed-evolution parameters for the Figure 5 strategy lookup.

In [ ]:
nk_strategy_N4 = load_raw(RAW_FILENAMES["nk_strategy_N4_A20"])
nk_decay_N4 = load_raw(RAW_FILENAMES["nk_decay_N4_A20"])

decay_rates, optimal_splits, optimal_base_chances = optimal_de_strategies(nk_strategy_N4, nk_decay_N4, np.array([[4, 1], [4, 2], [4, 3]]))
save_processed((decay_rates, optimal_splits, optimal_base_chances), "optimal_DE_strategies.pkl")

reshaped_strategies = nk_strategy_N4.reshape(100, -1, 300)
save_processed((reshaped_strategies[19, :, :], reshaped_strategies[14, :, :]), "NK_strategy_spaces.pkl")


## Plot Support Outputs

Derived processed files used by the visualisation notebook.


In [ ]:
save_processed(smooth_rugged_example(decay_grid), "smooth_rugged_example.pkl")
save_processed(nk_metric_comparison_from_accuracy(*ruggedness_accuracy), "NK_ruggedness_metric_comparison.pkl")
save_processed(strategy_prediction_summary(decay_rates, optimal_splits, optimal_base_chances), "strategy_prediction_accuracy.pkl")
save_processed(nk_de_summary(decay_grid), "NK_DE.pkl")
save_processed(fourier_analysis_summary(landscapes), "fourier_analysis.pkl")


## Empirical Strategy Inputs

In [ ]:
empirical_strategy = {
    name: load_raw(RAW_FILENAMES[f"empirical_strategy_{name}_uniform"])
    for name in ("GB1", "TrpB", "TEV", "ParD3")
}
empirical_popsize = {
    name: load_raw(RAW_FILENAMES[f"empirical_decay_{name}_popsize"])
    for name in ("GB1", "TrpB", "TEV", "ParD3")
}

def _strategy_selection_payload(name):
    decay = empirical_decay_uniform[name]
    sweep = empirical_strategy[name]
    decay_mean = (decay ** 2).mean(axis=(0, 1, 2))
    decay_mean = decay_mean / decay_mean[0]
    return (np.arange(decay_mean.shape[0]), decay_mean, None, sweep, None, None, None, None, decay)

for name in ("GB1", "TrpB", "TEV", "ParD3"):
    save_processed(_strategy_selection_payload(name), f"{name}_strategy_selection.pkl")

save_processed((None, None, None, np.asarray(nk_strategy_N4).mean(axis=0)), "empirical_lookup.pkl")


## Generation-Count Strategy Sweeps

In [ ]:
generation_strategy_sweeps = {
    steps: load_raw(RAW_FILENAMES[f"nk_strategy_N4_A20_steps{steps}"])
    for steps in (5, 25, 50, 75, 100, 500, 1000)
}
save_processed(generation_strategy_sweeps, "generation_strategy_sweeps.pkl")
